In [ ]:
!apt install swig cmake ffmpeg xvfb python3-opengl

In [ ]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
""")

!pip install pyvirtualdisplay imageio[ffmpeg]

%env MUJOCO_GL=egl
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
# Prepare to load data from google drive
from google.colab import drive
import datetime

# CONNECT TO GOOGLE DRIVE
gdrive_path = '/content/drive'
drive.mount(gdrive_path)

# DEFINE WORK DIRECTORY
current_step = 'step_005'
# workDir = f'{gdrive_path}/My Drive/Research/{current_step}'
workDir = os.path.join(gdrive_path, 'My Drive', 'Research', current_step)
print('WorkDir:', workDir)

log_dir = os.path.join(workDir, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
print('LogDir:', log_dir)

# create folder if it doesn't exists
if not os.path.exists(log_dir):
  os.makedirs(log_dir)

tf_log_dir = os.path.join(workDir, 'tf_logs')
print('TfLogDir:', tf_log_dir)

# create folder if it doesn't exists
if not os.path.exists(tf_log_dir):
  os.makedirs(tf_log_dir)

In [ ]:
# clean content folder
import os
import shutil

# location
location = "/content"

# directories
dirs = ["sample_data", "rl-zoo", "gym_darwin_op3", "videos"]

for dir in dirs:
    path = os.path.join(location, dir)
    try:
        shutil.rmtree(path)
    except OSError as e:
        print("Error: %s : %s" % (path, e.strerror))

In [ ]:
# Install Darwin Model
model_path = '/content/gym_darwin_op3'

if os.path.isdir(model_path):
  print(f"The directory '{model_path}' exists - git pull")
  %cd {model_path}
  !git pull
  %cd /
else:
  print(f"The directory '{model_path}' does not exist - git clone")
  !git clone --single-branch --branch {current_step} https://github.com/Gianzanti/robofei_mestrado.git {model_path}


In [ ]:
!pip install -e {model_path}

In [ ]:
# Install RL Zoo
trainner_path = '/content/rl-zoo'

if os.path.isdir(trainner_path):
  print(f"The directory '{trainner_path}' exists - git pull")
  %cd {trainner_path}
  !git pull
  %cd /
else:
  print(f"The directory '{trainner_path}' does not exist - git clone")
  !git clone https://github.com/Gianzanti/rl-zoo.git {trainner_path}


In [ ]:
!pip install -e {trainner_path}

In [ ]:
# Hyper Parameters Tunning
%cd {trainner_path}

algos_cpu = ['ppo', 'a2c']
algos_cuda = ['ddpg', 'sac', 'td3']

n_timestep = 5_000_000
save_freq = min(50_000, int(n_timestep / 10))
eval_freq = min(100_000, int(n_timestep / 10))

max_episode_steps = 1000
wrapper = [{"gymnasium.wrappers.TimeLimit": {"max_episode_steps": max_episode_steps}}]

for algo in algos_cpu:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    ctrl_cost_weight:1e-3 target_distance:2.0 forward_velocity_weight:1.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cpu
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 2000 \
    --load-best -o "{log_dir}" -f "{log_dir}"

for algo in algos_cuda:
  print('Training:', algo)
  config = f'research_config/{algo}.yml'

  !python3 train.py --algo {algo} --env DarwinOp3-v2 -conf {config} -f "{log_dir}"\
    --tensorboard-log "{tf_log_dir}" --save-freq {save_freq} \
    --vec-env subproc --eval-freq {eval_freq} --n-eval-envs 1 --eval-episodes 10 \
    --env-kwargs keep_alive_reward:0.5 motor_max_torque:3.0 \
    ctrl_cost_weight:1e-3 target_distance:2.0 forward_velocity_weight:1.0 \
    --hyperparams n_timesteps:{n_timestep} env_wrapper:"{wrapper}"\
    --device cuda
  !python -m rl_zoo3.record_video --algo {algo} --env DarwinOp3-v2 -n 2000 \
    --load-best -o "{log_dir}" -f "{log_dir}"



Streaming output truncated to the last 5000 lines.
|    fps             | 846      |
|    time_elapsed    | 846      |
|    total_timesteps | 716712   |
| train/             |          |
|    actor_loss      | -2.84    |
|    critic_loss     | 0.0431   |
|    learning_rate   | 0.001    |
|    n_updates       | 88338    |
---------------------------------
----------------------------------
| mean_episode/      |           |
|    control_cost    | 0.101     |
|    health_reward   | 0.5       |
|    pos_x           | -0.000436 |
|    pos_y           | 0.0144    |
|    pos_z           | 0.283     |
|    vel_x           | 0.0536    |
|    vel_y           | -0.037    |
| rollout/           |           |
|    ep_len_mean     | 187       |
|    ep_rew_mean     | 75.3      |
| time/              |           |
|    episodes        | 15060     |
|    fps             | 846       |
|    time_elapsed    | 848       |
|    total_timesteps | 718600    |
| train/             |           |
|    actor_lo